In [5]:
from QUANTAXIS import QA_fetch_get_stock_list
from pymongo import MongoClient
from pymongo.operations import InsertOne
from datetime import datetime
import time

In [25]:
from QUANTAXIS import QA_util_to_json_from_pandas,QA_fetch_get_stock_list,QA_util_get_trade_gap

In [22]:
mongodb_uri='mongodb://localhost:27017/'
database_name='stock_db'
source_collection='stock_min'
target_collection='stock_min_cn'
target_date='2026-08-08'
batch_size=10000

"""复制指定日期的数据到新集合"""

# 连接数据库
DATABASE = MongoClient(mongodb_uri)
# db = client[database_name]

In [23]:
def QA_SU_save_stock_min_super_optimized(client=DATABASE, ui_log=None, ui_progress=None):
    """超级优化版：批量保存所有股票的所有周期数据"""
    
    stock_list = QA_fetch_get_stock_list().code.unique().tolist()
    coll = client.stock_min
    
    # 确保索引存在
    coll.create_index([
        ('code', pymongo.ASCENDING),
        ('type', pymongo.ASCENDING),
        ('datetime', pymongo.ASCENDING)
    ], unique=True)
    
    err = []
    end_time = str(now_time())[0:19]
    
    def save_stock_batch(code, coll):
        code_str = str(code)[0:6]
        
        try:
            # 获取该股票所有周期的最新时间
            freq_types = ['1min', '5min', '15min', '30min', '60min']
            latest_times = {}
            
            for doc in coll.aggregate([
                {'$match': {'code': code_str, 'type': {'$in': freq_types}}},
                {'$sort': {'datetime': -1}},
                {'$group': {'_id': '$type', 'latest_time': {'$first': '$datetime'}}}
            ]):
                latest_times[doc['_id']] = doc['latest_time']
            
            # 批量获取所有周期数据（只获取一次1分钟数据）
            start_times = {}
            need_update = False
            
            for freq_type in freq_types:
                start_time = latest_times.get(freq_type, '2026-08-14')
                if start_time != end_time:
                    start_times[freq_type] = start_time
                    need_update = True
            
            if not need_update:
                return code_str
            
            # 获取所有周期数据
            all_data = QA_fetch_get_stock_min_batch(
                code_str, 
                min(start_times.values()), 
                end_time,
                list(start_times.keys())
            )
            
            # 批量保存
            if all_data:
                operations = []
                for freq_type, df in all_data.items():
                    if df.empty:
                        continue
                    
                    records = QA_util_to_json_from_pandas(df)
                    
                    # 过滤掉已存在的数据
                    if freq_type in latest_times:
                        latest = latest_times[freq_type]
                        records = [r for r in records if r['datetime'] > latest]
                    
                    if records:
                        for record in records:
                            operations.append(
                                pymongo.UpdateOne(
                                    {
                                        'code': record['code'],
                                        'datetime': record['datetime'],
                                        'type': record['type']
                                    },
                                    {'$set': record},
                                    upsert=True
                                )
                            )
                
                if operations:
                    # 批量执行，每1000条一批
                    batch_size = 1000
                    for i in range(0, len(operations), batch_size):
                        batch = operations[i:i+batch_size]
                        coll.bulk_write(batch, ordered=False)
            
            return code_str
            
        except Exception as e:
            QA_util_log_info(f"保存失败 {code_str}: {e}", ui_log=ui_log)
            err.append(code_str)
            return None
    
    # 使用线程池
    max_workers = min(4, len(stock_list))  # 减少并发数避免被限制
    executor = ThreadPoolExecutor(max_workers=max_workers)
    
    futures = {executor.submit(save_stock_batch, code, coll): code for code in stock_list}
    
    # 进度显示
    total = len(stock_list)
    completed = 0
    
    for future in concurrent.futures.as_completed(futures):
        completed += 1
        if completed % 10 == 0:  # 每10个更新一次进度
            progress = (completed / total) * 100
            QA_util_log_info(
                f'进度: {completed}/{total} ({progress:.1f}%)',
                ui_log=ui_log,
                ui_progress=ui_progress,
                ui_progress_int_value=int(progress * 100)
            )
    
    if err:
        QA_util_log_info(f'错误代码: {err}', ui_log=ui_log)
    else:
        QA_util_log_info('全部保存成功！', ui_log=ui_log)

In [ ]:
from functools import lru_cache
import hashlib

class StockMinCache:
    """分钟数据缓存"""
    
    def __init__(self, cache_size=128):
        self.cache = {}
        self.cache_size = cache_size
    
    def get_cache_key(self, code, start, end, freq):
        """生成缓存键"""
        key_str = f"{code}_{start}_{end}_{freq}"
        return hashlib.md5(key_str.encode()).hexdigest()
    
    def get(self, code, start, end, freq):
        """获取缓存数据"""
        key = self.get_cache_key(code, start, end, freq)
        return self.cache.get(key)
    
    def set(self, code, start, end, freq, data):
        """设置缓存"""
        key = self.get_cache_key(code, start, end, freq)
        if len(self.cache) >= self.cache_size:
            # 移除最早的数据
            self.cache.pop(next(iter(self.cache)))
        self.cache[key] = data

# 全局缓存实例
_data_cache = StockMinCache(cache_size=512)

def QA_fetch_get_stock_min_with_cache(code, start, end, frequence='1min', ip=None, port=None):
    """带缓存的获取函数"""
    
    # 检查缓存
    cached_data = _data_cache.get(code, start, end, frequence)
    if cached_data is not None:
        return cached_data
    
    # 获取数据
    data = QA_fetch_get_stock_min_optimized(code, start, end, frequence, ip, port)
    
    # 存入缓存
    if not data.empty:
        _data_cache.set(code, start, end, frequence, data)
    
    return data

In [1]:
def QA_fetch_get_stock_min_batch(code, start, end, freq_types=None, ip=None, port=None):
    """
    批量获取一个股票的所有周期数据
    一次性获取1分钟数据，然后聚合生成其他周期
    """
    if freq_types is None:
        freq_types = ['1min', '5min', '15min', '30min', '60min']
    
    code_str = str(code)[0:6]
    start_date = str(start)[0:10]
    end_date = str(end)[0:10]
    
    # 1. 只获取1分钟原始数据
    freq_code = 8  # 1分钟
    trade_days = QA_util_get_trade_gap(start_date, end_date)
    total_records = 240 * trade_days
    
    if total_records > 20800:
        total_records = 20800
    
    api = get_connection(ip, port)
    
    # 获取1分钟数据
    chunks = []
    remaining = total_records
    offset = 0
    
    while remaining > 0:
        chunk_size = min(800, remaining)
        chunk = api.get_security_bars(freq_code, _select_market_code(code_str), code_str, offset, chunk_size)
        if chunk:
            chunks.append(api.to_df(chunk))
        remaining -= chunk_size
        offset += chunk_size
    
    if not chunks:
        return {}
    
    df_1min = pd.concat(chunks, axis=0, sort=False)
    if df_1min.empty:
        return {}
    
    # 处理1分钟数据
    df_1min['datetime'] = pd.to_datetime(df_1min['datetime'], utc=False)
    df_1min = df_1min.set_index('datetime').sort_index()
    df_1min = df_1min[start:end]
    
    if df_1min.empty:
        return {}
    
    # 2. 从1分钟数据聚合生成其他周期
    result = {}
    
    # 保存1分钟数据
    df_1min_copy = df_1min.copy()
    df_1min_copy = df_1min_copy.assign(
        code=code_str,
        date=df_1min_copy.index.strftime('%Y-%m-%d'),
        date_stamp=df_1min_copy.index.map(lambda x: QA_util_date_stamp(x)),
        time_stamp=df_1min_copy.index.map(lambda x: QA_util_time_stamp(x)),
        type='1min'
    )
    df_1min_copy['datetime'] = df_1min_copy.index.strftime('%Y-%m-%d %H:%M:%S')
    result['1min'] = df_1min_copy.reset_index(drop=True)
    
    # 聚合生成其他周期
    freq_agg_map = {
        '5min': ('5min', 5),
        '15min': ('15min', 15),
        '30min': ('30min', 30),
        '60min': ('60min', 60)
    }
    
    for freq_type, (label, minutes) in freq_agg_map.items():
        if freq_type not in freq_types:
            continue
            
        # 使用resample进行聚合
        resampled = df_1min.resample(f'{minutes}T').agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum',
            'amount': 'sum'
        }).dropna()
        
        # 只保留交易时间的数据（去除空值）
        resampled = resampled.between_time('09:30', '15:00')
        
        if not resampled.empty:
            df = resampled.assign(
                code=code_str,
                date=resampled.index.strftime('%Y-%m-%d'),
                date_stamp=resampled.index.map(lambda x: QA_util_date_stamp(x)),
                time_stamp=resampled.index.map(lambda x: QA_util_time_stamp(x)),
                type=label
            )
            df['datetime'] = df.index.strftime('%Y-%m-%d %H:%M:%S')
            result[label] = df.reset_index(drop=True)
    
    return result

In [14]:
import time

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    if hours > 0:
        return f"{hours}时{minutes}分{secs}秒"
    elif minutes > 0:
        return f"{minutes}分{secs}秒"
    else:
        return f"{secs}秒"

In [24]:
start_time = time.time()
QA_SU_save_stock_min_super_optimized()
elapsed = time.time() - start_time
print(f"⏱️ 保存分钟数据完成，总耗时: {format_time(elapsed)}")

TypeError: QA_fetch_get_stock_list() missing 1 required positional argument: 'package'